# Generate calorimeter showers and plot them

Rebuilds a pretrained **humite** model from its bundled config, generates showers
with `generate_showers` (the same code path as the `humite-generate` CLI), and
compares them to Geant4 using the project's `plot_features`.

**Requirements:** run inside the humite environment (Singularity/Docker image or
`pip install -e .`), ideally with a GPU. Download checkpoints first:
`cd checkpoints && ./download_checkpoints.sh`.

In [ ]:
%matplotlib inline
import random
from pathlib import Path

import numpy as np
import torch

from humite.cli.generate import generate_showers
from humite.core.callbacks.generative_eval_helpers import physical_to_ak
from humite.core.plotting.plotting import plot_features

# Start Jupyter from the repo root (folder with pyproject.toml) so `humite` imports.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

# Pick a checkpoint; the config.yaml next to it (architecture, sampling, filter) loads automatically.
CKPT = REPO / "checkpoints" / "spade" / "getting_square_x1" / "model.ckpt"
assert CKPT.is_file(), f"missing checkpoint {CKPT} - run checkpoints/download_checkpoints.sh"

print("repo:", REPO, "| checkpoint:", CKPT.relative_to(REPO))

## 1. Generate showers

In [ ]:
out = generate_showers(
    checkpoint=str(CKPT),  # config.yaml next to it is found automatically
    n_showers=512,
    energy_min_gev=10.0,
    energy_max_gev=100.0,
    seed=42,
)
mask = out.get("hit_mask")
if mask is None:
    mask = out["hit_features"][..., 3] > 0
n_hits = mask.astype(bool).sum(1)
print(
    f"generated {out['hit_features'].shape[0]} showers; hits/shower: {n_hits.min()}-{n_hits.max()}"
)

## 2. Build feature arrays (and optionally load Geant4)

In [ ]:
def to_ak(hit_features, hit_mask, incident):
    return physical_to_ak(
        torch.as_tensor(hit_features), torch.as_tensor(hit_mask).bool(), torch.as_tensor(incident)
    )


gen_phys, gen_global, _ = to_ak(out["hit_features"], mask, out["incident_energy"])
arrays_global = {"SPADE": gen_global}
arrays_phys = {"SPADE": gen_phys}

print("datasets:", list(arrays_global))

## 3. Number of hits, energy response, and hit-energy spectrum

In [ ]:
colors = [{"Geant4": "#94A3B8", "SPADE": "#818CF8"}.get(k, None) for k in arrays_global]
ratio = len(arrays_global) > 1

# Per-shower (global) features
fig, _ = plot_features(
    arrays_global,
    names={
        "n_hits": "Number of hits",
        "energy_sum_over_incident_energy": "Energy sum / incident energy",
    },
    bins_dict={
        "n_hits": np.linspace(0, 3700, 40),
        "energy_sum_over_incident_energy": np.linspace(0, 0.00004, 35),
    },
    flatten=False,
    ratio=ratio,
    colors=colors,
    ax_size=(4, 3),
)

# Per-hit (point cloud) feature - energy spectrum
fig, _ = plot_features(
    arrays_phys,
    names={"energy": "Hit energy [MeV]"},
    bins_dict={"energy": np.geomspace(0.1, 100, 50)},
    logscale_xaxis_features=["energy"],
    ratio=ratio,
    colors=colors,
    ax_size=(4, 3),
)